本章我们来采用策略迭代与价值迭代来求解悬崖漫步（Cliff Walking）这个环境中的最优策略

它要求一个智能体从起点出发，避开悬崖行走，最终到达目标位置；有一个 4×12 的网格世界，每一个网格表示一个状态。智能体的起点是左下角的状态，目标是右下角的状态，智能体在每一个状态都可以采取 4 种动作：上、下、左、右。如果智能体采取动作后触碰到边界墙壁则状态不发生改变，否则就会相应到达下一个状态。环境中有一段悬崖，智能体掉入悬崖或到达目标状态都会结束动作并回到起点，也就是说掉入悬崖或者达到目标状态是终止状态。智能体每走一步的奖励是 −1，掉入悬崖的奖励是 −100。

In [ ]:
import copy

class CliffWalkingEnv:
    """
        悬崖漫步环境
    """
    def __init__(self, ncol=12, nrow=4):
        """
            初始化环境
            ncol: 列数
            nrow: 行数
        """
        self.ncol = ncol  # 定义网格世界的列
        self.nrow = nrow  # 定义网格世界的行

        # 转移矩阵P(s,a) = [(p, next_state, reward, done)]包含下一个状态和奖励
        self.P = self.createP()

    def createP(self):
        """
            创建转移矩阵P(s,a) = [(p, next_state, reward, done)]
            先来划定危险和结束区域：如果下一步的 y 是 3（最后一行），并且 x 大于 0（排除了第 0 列）。只要进了这个区域，游戏就直接结束
            界定悬崖位置和终点：如果下一列不是最后一列（即不是第11列），那么奖励就是 -100；如果下一列是最后一列（即第11列），奖励就是 0。
        """
        # 外层循环: i 遍历 48 个状态 (4*12)。内层循环: 给每个状态分配一个包含 4 个空列表的列表 (对应4个动作)
        # 最终 P 是一个拥有 48 个元素的列表，每个元素里又包含 4 个子列表
        P = [[[] for j in range(4)] for i in range(self.nrow * self.ncol)]
        
        # 定义 4 种动作导致的坐标变化量 [x轴变化, y轴变化]
        # 注意：这里的坐标系原点 (0,0) 在左上角，向右 x 增加，向下 y 增加。
        # change[0]: 上 (y减1) -> [0, -1]
        # change[1]: 下 (y加1) -> [0, 1]
        # change[2]: 左 (x减1) -> [-1, 0]
        # change[3]: 右 (x加1) -> [1, 0]
        change = [[0, -1], [0, 1], [-1, 0], [1, 0]]

        # 嵌套循环：遍历网格中的每一个位置 (i 代表行/y坐标，j 代表列/x坐标)
        for i in range(self.nrow):
            for j in range(self.ncol):
                # 遍历在当前位置 (i, j) 下可以执行的 4 种动作 (a 取值 0, 1, 2, 3)
                for a in range(4):
                    # --- 特殊情况 1：智能体当前已经处于“悬崖”或“终点” ---
                    # 如果行数 i == 3 (最后一行) 且列数 j > 0 (排除了 j=0 的起点)
                    if i == self.nrow - 1 and j > 0:
                        # P[一维状态索引][动作] = [(转移概率, 下一个状态, 奖励, 是否结束)]
                        # 一维状态索引计算法: 行号 * 总列数 + 列号 (即 i * self.ncol + j)
                        # 因为已经在终点或掉入悬崖，游戏已结束，所以无论做什么动作：
                        # 概率是 1，状态留在原地不发生变化，奖励为 0，done 为 True(结束)
                        P[i * self.ncol + j][a] = [(1, i * self.ncol + j, 0, True)]     # 以一列为一个状态，行数 * 列数 + 列数 = 状态索引

                        # 直接跳过当前动作的后续计算，进入下一个动作/状态的循环
                        continue 

                    # 其他位置
                    next_x = min(self.ncol - 1, max(0, j + change[a][0]))
                    next_y = min(self.nrow - 1, max(0, i + change[a][1]))
                    next_state = next_y * self.ncol + next_x
                    # 每一步的奖励
                    reward = -1
                    # 
                    done = False
                    # 下一个位置在悬崖或者终点（下一步的行数为最后一行且列数大于0）
                    if next_y == self.nrow - 1 and next_x > 0:
                        done = True
                        # 下一个位置在悬崖
                        if next_x != self.ncol - 1:  
                            reward = -100

                    # 完善转移矩阵 P 的内容：对于当前状态 (i, j) 和动作 a，记录转移概率、下一个状态、奖励和是否结束
                    P[i * self.ncol + j][a] = [(1, next_state, reward, done)]

        return P    

策略迭代算法的代码实现过程

In [ ]:
class PolicyIteration:
    """ 
        策略迭代算法 
    """
    def __init__(self, env, theta, gamma):
        """
            env：环境对象
            theta：策略评估收敛阈值
            gamma：折扣因子
            theta：策略评估收敛阈
        """
        self.env = env
        # 初始化价值为0(大小为状态数)
        self.v = [0] * self.env.ncol * self.env.nrow  
        # 初始化为均匀随机策略（每个状态选择每个动作都是一样的） 
        self.pi = [[0.25, 0.25, 0.25, 0.25]
                   for i in range(self.env.ncol * self.env.nrow)]  
        # 策略评估收敛阈，因为我们使用迭代解求解，找到最终的 0 点是比较难的；所以我们认为前后的差小于一个值即认为收敛了
        self.theta = theta
        # 折扣因子
        self.gamma = gamma

    def policy_evaluation(self):
        """
            策略评估
        """
        # 策略评估迭代次数
        cnt = 1
        while 1:
            # 用来记录这一轮迭代中，所有状态里价值变化的最大幅度。这是判断是否收敛的关键指标
            max_diff = 0
            # 用于存放迭代后的状态价值
            new_v = [0] * self.env.ncol * self.env.nrow

            # 对每一个状态进行迭代求解
            for s in range(self.env.ncol * self.env.nrow):
                # 开始计算状态s下的所有Q(s,a)价值
                qsa_list = [] 
                for a in range(4):
                    # 初始化当前动作 a 的动作价值为 0
                    qsa = 0
                    # 环境的状态转移模型（详细看上面的定义）
                    for res in self.env.P[s][a]:
                        p, next_state, r, done = res
                        qsa += p * (r + self.gamma * self.v[next_state] * (1 - done))
